## 1. Environment Preparation

Install Unsloth and updated HuggingFace libraries for Mistral support.

In [ ]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")

## 2. Load Dataset & Format for Instruction Tuning

Load the Stoic QA dataset and apply Mistral's chat template.

In [ ]:
from datasets import load_dataset

# Load instruct dataset (Stoic QA; swap for Biblical like "facat/Socratic")
dataset = load_dataset("Manel/Reddit_Stoicism_QA_610", split="train")

print(f"✓ Dataset loaded: {len(dataset)} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\n--- First Example ---")
for key, value in dataset[0].items():
    print(f"{key}: {str(value)[:200]}")  # Show first 200 chars of each field

# Shuffle and subset for testing
dataset = dataset.shuffle(seed=42).select(range(min(1000, len(dataset))))
print(f"\n✓ Using {len(dataset)} examples for training")

## 3. Load Model & Tokenizer with Unsloth

Load the quantized Mistral-7B-Instruct model with 4-bit precision and configure the tokenizer.

In [ ]:
from unsloth import FastLanguageModel
import torch
from transformers import BitsAndBytesConfig

model_name = "unsloth/mistral-7b-instruct-v0.3"
max_seq_length = 2048

# Configure 4-bit quantization with CPU offload
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    quantization_config=bnb_config,
    device_map={"": 0}  # Force all on GPU 0
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {model_name}")
print(f"✓ Tokenizer configured")
print(f"✓ Max sequence length: {max_seq_length}")

In [ ]:
# Format dataset using Mistral chat template WITH SYSTEM PROMPT
def format_instruct(example):
    # Add system prompt to make model respond AS a Stoic philosopher
    system_prompt = "You are a Stoic philosopher, responding with wisdom in the tradition of Marcus Aurelius, Epictetus, and Seneca. Speak in first person, sharing Stoic principles through personal reflection and direct counsel."
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example['instruction']},
        {"role": "assistant", "content": example['response']}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False
    )
    return {"text": text}

# Format and remove old columns
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

print(f"✓ Dataset formatted: {len(dataset)} examples")
print(f"\n--- Formatted Example ---")
print(dataset[0]['text'][:500])

## 4. Add LoRA Adapters

Configure LoRA for efficient fine-tuning with attention and MLP projection layers.

In [ ]:
from peft import LoraConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length
)

print("LoRA adapters added successfully")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Trainer Setup & Training

**Key improvements:**
- Cosine LR decay (better than linear)
- 500 steps instead of 100 (dataset has 610 examples, need multiple epochs)
- System prompt added to force first-person Stoic responses

**Dataset limitation:** This Reddit dataset has third-person advice ABOUT stoicism. For true first-person Stoic responses, consider:
- `teknium/OpenHermes-2.5` filtered for philosophy
- Custom dataset from Meditations/Enchiridion with synthetic Q&A
- Fine-tune on Marcus Aurelius' actual writings

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling

# Create data collator
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=collator,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=500,  # Increased from 100 - more training needed
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",  # Changed to cosine decay
        seed=3407,
        output_dir="./stoic_instruct_finetune",
        report_to="none"
    )
)

print("✓ Trainer configured with cosine LR schedule, 500 steps")

In [ ]:
# Start training
trainer.train()

## 7. Save Model & Inference

Save the fine-tuned model and test inference with a Stoic question.

In [ ]:
# Save merged model
model.save_pretrained_merged("./stoic_instruct_model", tokenizer, save_method="merged_16bit")

print("Model saved to ./stoic_instruct_model")

In [ ]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test inference
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": "How to apply Stoic principles?"}],
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.7, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)

print("\n=== Inference Test ===")
print(response)

## Notes

### For Biblical/Socratic Style:
Replace dataset in cell 2:
```python
dataset = load_dataset("facat/Socratic", split="train")
```

Adjust `format_instruct` function to match dataset columns (e.g., 'question'/'response', 'prompt'/'completion', etc.)

### Troubleshooting:
- If DGX hangs: Reduce `max_steps` or `per_device_train_batch_size`
- Check dataset columns match expected format in `format_instruct`
- For memory issues: Reduce `max_seq_length` or enable `packing=True`